# ETL Completo con PySpark - Análisis de Sentimientos de Stock

Este notebook implementa un proceso ETL completo (Extract, Transform, Load) utilizando **PySpark** para analizar datos de sentimientos relacionados con acciones del mercado financiero.

## 📋 Contenido del Notebook:
1. **Instalación de Librerías** - Configuración del entorno PySpark
2. **Importación y Configuración** - Inicialización de Spark Session
3. **Extracción de Datos** - Carga desde archivos CSV
4. **Exploración de Datos** - Análisis inicial del dataset
5. **Limpieza y Transformación** - Procesamiento con PySpark
6. **Validación de Datos** - Verificación de calidad
7. **Visualizaciones** - Gráficas con matplotlib y seaborn
8. **Carga Final** - Guardado en múltiples formatos

---
**Autor:** Javier Plata  
**Dataset:** Stock Sentiment Analysis  
**Fecha:** Octubre 2025

## 📦 1. Instalación de Librerías Requeridas

Primero instalamos todas las librerías necesarias para nuestro análisis ETL con PySpark.

In [ ]:
# Instalación de PySpark y librerías adicionales
import subprocess
import sys

def install_package(package):
    """Función para instalar paquetes si no están disponibles"""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ {package} instalado correctamente")
    except Exception as e:
        print(f"❌ Error instalando {package}: {e}")

# Lista de paquetes necesarios
packages = [
    "pyspark==3.4.1",           # PySpark para procesamiento distribuido
    "findspark",                # Para encontrar la instalación de Spark
    "matplotlib==3.7.2",       # Para visualizaciones básicas
    "seaborn==0.12.2",         # Para visualizaciones estadísticas
    "pandas==2.0.3",           # Para manipulación de datos
    "numpy==1.24.3",           # Para operaciones numéricas
    "plotly==5.15.0",          # Para gráficas interactivas
    "openpyxl==3.1.2"          # Para leer archivos Excel
]

print("🚀 Iniciando instalación de dependencias...")
print("="*60)

for package in packages:
    install_package(package)

print("\n✨ Instalación completada!")
print("🔄 Reinicia el kernel si es la primera vez que instalas PySpark")

## 🔧 2. Importación de Librerías e Inicialización de Spark Session

Importamos todas las librerías necesarias y configuramos una sesión de Spark optimizada para nuestro análisis.

In [ ]:
# Configuración de findspark (para encontrar Spark automáticamente)
try:
    import findspark
    findspark.init()
    print("✅ FindSpark configurado correctamente")
except ImportError:
    print("⚠️  FindSpark no disponible, asegúrate de que Spark esté en el PATH")

# Importaciones de PySpark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Importaciones para análisis y visualización
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configuraciones adicionales
import warnings
import os
from datetime import datetime, date

# Suprimir warnings para output más limpio
warnings.filterwarnings('ignore')

# Configurar matplotlib para mejor visualización
plt.style.use('default')
sns.set_palette("husl")

print("✅ Todas las librerías importadas correctamente")
print("📊 Configuraciones de visualización aplicadas")

In [ ]:
# Crear Spark Session con configuración optimizada
print("🚀 Iniciando Spark Session...")

spark = SparkSession.builder \
    .appName("ETL_Stock_Sentiment_Analysis") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.minPartitionSize", "1MB") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .getOrCreate()

# Configurar el nivel de log para reducir verbosidad
spark.sparkContext.setLogLevel("WARN")

print("✅ Spark Session iniciada correctamente!")
print(f"🔧 Versión de Spark: {spark.version}")
print(f"💻 Aplicación: {spark.sparkContext.appName}")
print(f"🌐 Master: {spark.sparkContext.master}")
print(f"📊 Particiones por defecto: {spark.conf.get('spark.sql.shuffle.partitions')}")

# Verificar configuración de Spark
print("\n📋 Configuración de Spark Session:")
print("-" * 50)
important_configs = [
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.shuffle.partitions"
]

for config in important_configs:
    value = spark.conf.get(config)
    print(f"   {config}: {value}")
    
print("\n🎯 Spark Session listo para procesar datos!")

## 📁 3. Extracción de Datos (Extract)

Cargamos el dataset de análisis de sentimientos de stock usando PySpark para aprovechar su capacidad de procesamiento distribuido.

In [ ]:
# Definir rutas de archivos
input_csv_path = "Files/stock_senti_analysis.csv"
output_directory = "Files/"

print("📂 EXTRAYENDO DATOS CON PYSPARK")
print("=" * 50)

try:
    # Verificar si el archivo existe
    if os.path.exists(input_csv_path):
        print(f"✅ Archivo encontrado: {input_csv_path}")
        
        # Cargar datos usando PySpark con inferencia automática de schema
        print("🔄 Cargando datos en Spark DataFrame...")
        
        df_spark = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .option("multiline", "true") \
            .option("escape", '"') \
            .csv(input_csv_path)
        
        print("✅ Datos cargados exitosamente en Spark DataFrame!")
        
        # Información básica del DataFrame
        print(f"\n📊 INFORMACIÓN BÁSICA DEL DATASET:")
        print(f"   • Número de filas: {df_spark.count():,}")
        print(f"   • Número de columnas: {len(df_spark.columns)}")
        print(f"   • Número de particiones: {df_spark.rdd.getNumPartitions()}")
        
        # Mostrar esquema del DataFrame
        print(f"\n🏗️  ESQUEMA DEL DATAFRAME:")
        df_spark.printSchema()
        
    else:
        print(f"❌ Error: Archivo no encontrado en {input_csv_path}")
        print("💡 Asegúrate de que el archivo esté en la ubicación correcta")
        
except Exception as e:
    print(f"❌ Error al cargar los datos: {str(e)}")
    print("💡 Verifica la ruta del archivo y el formato CSV")

In [ ]:
# Mostrar las primeras filas del dataset
print("👀 MUESTRA DE LOS DATOS CARGADOS:")
print("=" * 50)

# Mostrar primeras 5 filas
print("📋 Primeras 5 filas:")
df_spark.show(5, truncate=False)

# Mostrar nombres de columnas
print(f"\n📝 COLUMNAS DEL DATASET ({len(df_spark.columns)} total):")
for i, col in enumerate(df_spark.columns, 1):
    print(f"   {i}. {col}")
    
# Obtener estadísticas básicas para columnas numéricas
print(f"\n📊 ESTADÍSTICAS DESCRIPTIVAS:")
df_spark.describe().show()

print("✅ Extracción de datos completada correctamente!")

## 🔍 4. Exploración y Perfilado de Datos

Realizamos un análisis exploratorio detallado para entender la estructura, calidad y características del dataset.

In [ ]:
# Análisis detallado de calidad de datos
print("🔍 ANÁLISIS EXPLORATORIO DE DATOS")
print("=" * 60)

# 1. Análisis de valores nulos
print("1️⃣ ANÁLISIS DE VALORES NULOS:")
print("-" * 40)

null_counts = []
total_rows = df_spark.count()

for column in df_spark.columns:
    null_count = df_spark.filter(col(column).isNull()).count()
    null_percentage = (null_count / total_rows) * 100
    null_counts.append((column, null_count, null_percentage))
    print(f"   {column}: {null_count:,} nulos ({null_percentage:.2f}%)")

# 2. Tipos de datos detallados
print(f"\n2️⃣ ANÁLISIS DE TIPOS DE DATOS:")
print("-" * 40)
for field in df_spark.schema.fields:
    print(f"   {field.name}: {field.dataType} (Nullable: {field.nullable})")

# 3. Análisis de duplicados
print(f"\n3️⃣ ANÁLISIS DE DUPLICADOS:")
print("-" * 40)
total_rows = df_spark.count()
unique_rows = df_spark.distinct().count()
duplicate_count = total_rows - unique_rows
duplicate_percentage = (duplicate_count / total_rows) * 100

print(f"   Filas totales: {total_rows:,}")
print(f"   Filas únicas: {unique_rows:,}")
print(f"   Duplicados: {duplicate_count:,} ({duplicate_percentage:.2f}%)")

# 4. Análisis de la columna objetivo (Label)
print(f"\n4️⃣ ANÁLISIS DE LA VARIABLE OBJETIVO (Label):")
print("-" * 40)
if 'Label' in df_spark.columns:
    label_distribution = df_spark.groupBy('Label').count().collect()
    for row in label_distribution:
        percentage = (row['count'] / total_rows) * 100
        sentiment = "Negativo" if row['Label'] == 0 else "Positivo"
        print(f"   {sentiment} ({row['Label']}): {row['count']:,} ({percentage:.2f}%)")
else:
    print("   ⚠️  Columna 'Label' no encontrada")

print(f"\n✅ Análisis exploratorio completado!")

In [ ]:
# Análisis específico de columnas de texto y fecha
print("🔬 ANÁLISIS ESPECÍFICO DE COLUMNAS")
print("=" * 60)

# 5. Análisis de la columna de fecha (si existe)
print("5️⃣ ANÁLISIS DE FECHAS:")
print("-" * 40)
date_columns = [col_name for col_name in df_spark.columns 
               if any(keyword in col_name.lower() for keyword in ['date', 'fecha', 'time'])]

if date_columns:
    for date_col in date_columns:
        print(f"   Columna de fecha encontrada: {date_col}")
        # Mostrar rango de fechas si es posible
        try:
            df_spark.select(min(date_col).alias("min_date"), 
                          max(date_col).alias("max_date")).show()
        except:
            print(f"   ⚠️  No se pudo analizar como fecha: {date_col}")
else:
    print("   ⚠️  No se encontraron columnas de fecha")

# 6. Análisis de columnas de texto
print(f"\n6️⃣ ANÁLISIS DE COLUMNAS DE TEXTO:")
print("-" * 40)
text_columns = [col_name for col_name in df_spark.columns 
               if any(keyword in col_name.lower() for keyword in ['text', 'headline', 'title', 'content'])]

if text_columns:
    for text_col in text_columns:
        print(f"\n   📝 Análisis de '{text_col}':")
        
        # Longitud promedio del texto
        df_with_length = df_spark.withColumn(f"{text_col}_length", 
                                           length(col(text_col)))
        
        avg_length = df_with_length.agg(avg(f"{text_col}_length").alias("avg_length")).collect()[0]["avg_length"]
        max_length = df_with_length.agg(max(f"{text_col}_length").alias("max_length")).collect()[0]["max_length"]
        min_length = df_with_length.agg(min(f"{text_col}_length").alias("min_length")).collect()[0]["min_length"]
        
        print(f"      • Longitud promedio: {avg_length:.1f} caracteres")
        print(f"      • Longitud mínima: {min_length} caracteres")
        print(f"      • Longitud máxima: {max_length} caracteres")
        
        # Mostrar algunos ejemplos
        print(f"      • Ejemplos de texto:")
        sample_texts = df_spark.select(text_col).limit(3).collect()
        for i, row in enumerate(sample_texts, 1):
            text_sample = row[text_col][:100] + "..." if len(str(row[text_col])) > 100 else row[text_col]
            print(f"         {i}. {text_sample}")
else:
    print("   ⚠️  No se encontraron columnas de texto específicas")

print(f"\n✅ Análisis específico de columnas completado!")

## 🧹 5. Limpieza y Transformación de Datos

Aplicamos transformaciones y limpieza de datos usando las funciones nativas de PySpark para optimizar el rendimiento.

In [ ]:
# Inicio del proceso de transformación
print("🧹 LIMPIEZA Y TRANSFORMACIÓN DE DATOS")
print("=" * 60)

# Crear una copia del DataFrame para las transformaciones
df_clean = df_spark

print("1️⃣ ELIMINANDO DUPLICADOS...")
print("-" * 40)
initial_count = df_clean.count()
df_clean = df_clean.distinct()
final_count = df_clean.count()
removed_duplicates = initial_count - final_count

print(f"   • Filas iniciales: {initial_count:,}")
print(f"   • Filas después de limpiar duplicados: {final_count:,}")
print(f"   • Duplicados eliminados: {removed_duplicates:,}")

# 2. Manejo de valores nulos
print(f"\n2️⃣ MANEJANDO VALORES NULOS...")
print("-" * 40)

# Eliminar filas donde todas las columnas importantes sean nulas
important_columns = []
if 'Label' in df_clean.columns:
    important_columns.append('Label')

text_cols = [col_name for col_name in df_clean.columns 
             if any(keyword in col_name.lower() for keyword in ['text', 'headline', 'title'])]
important_columns.extend(text_cols)

if important_columns:
    print(f"   • Columnas importantes identificadas: {important_columns}")
    
    # Eliminar filas donde TODAS las columnas importantes son nulas
    condition = None
    for col_name in important_columns:
        if condition is None:
            condition = col(col_name).isNotNull()
        else:
            condition = condition | col(col_name).isNotNull()
    
    df_clean = df_clean.filter(condition)
    
    rows_after_null_removal = df_clean.count()
    removed_null_rows = final_count - rows_after_null_removal
    
    print(f"   • Filas después de eliminar nulos críticos: {rows_after_null_removal:,}")
    print(f"   • Filas con nulos críticos eliminadas: {removed_null_rows:,}")
else:
    print("   • No se identificaron columnas críticas para nulos")

print(f"\n✅ Limpieza básica completada!")

In [ ]:
# Transformaciones específicas según el tipo de datos
print("3️⃣ APLICANDO TRANSFORMACIONES ESPECÍFICAS...")
print("-" * 40)

# Transformar fechas (si existen)
date_columns = [col_name for col_name in df_clean.columns 
               if any(keyword in col_name.lower() for keyword in ['date', 'fecha'])]

if date_columns:
    print(f"   📅 Procesando columnas de fecha: {date_columns}")
    
    for date_col in date_columns:
        # Convertir a timestamp si no lo está ya
        try:
            df_clean = df_clean.withColumn(date_col, to_timestamp(col(date_col)))
            
            # Crear columnas derivadas útiles
            df_clean = df_clean.withColumn(f"{date_col}_year", year(col(date_col))) \
                              .withColumn(f"{date_col}_month", month(col(date_col))) \
                              .withColumn(f"{date_col}_day", dayofmonth(col(date_col))) \
                              .withColumn(f"{date_col}_weekday", date_format(col(date_col), "EEEE")) \
                              .withColumn(f"{date_col}_is_weekend", 
                                        when(date_format(col(date_col), "u").isin([6, 7]), True).otherwise(False))
            
            print(f"     ✅ {date_col}: columnas derivadas creadas")
            
        except Exception as e:
            print(f"     ⚠️  Error procesando {date_col}: {str(e)}")

# Transformar columnas de texto
text_columns = [col_name for col_name in df_clean.columns 
               if any(keyword in col_name.lower() for keyword in ['text', 'headline', 'title'])]

if text_columns:
    print(f"\n   📝 Procesando columnas de texto: {text_columns}")
    
    for text_col in text_columns:
        try:
            # Crear métricas de texto
            df_clean = df_clean.withColumn(f"{text_col}_length", length(col(text_col))) \
                              .withColumn(f"{text_col}_word_count", 
                                        size(split(col(text_col), " "))) \
                              .withColumn(f"{text_col}_clean", 
                                        regexp_replace(col(text_col), "[^a-zA-Z0-9\\s]", ""))
            
            print(f"     ✅ {text_col}: métricas de texto creadas")
            
        except Exception as e:
            print(f"     ⚠️  Error procesando {text_col}: {str(e)}")

# Crear características adicionales
print(f"\n4️⃣ CREANDO CARACTERÍSTICAS ADICIONALES...")
print("-" * 40)

# Normalizar la variable objetivo si existe
if 'Label' in df_clean.columns:
    print(f"   🎯 Normalizando variable objetivo 'Label'")
    df_clean = df_clean.withColumn("Label_normalized", 
                                  when(col("Label") == 1, "Positive")
                                  .when(col("Label") == 0, "Negative")
                                  .otherwise("Unknown"))

# Crear un ID único para cada fila
df_clean = df_clean.withColumn("row_id", monotonically_increasing_id())

print(f"   ✅ ID único añadido a cada fila")
print(f"\n✅ Transformaciones completadas!")

# Mostrar el esquema actualizado
print(f"\n📋 ESQUEMA DEL DATAFRAME TRANSFORMADO:")
print("-" * 40)
df_clean.printSchema()

## ✅ 6. Validación de Datos Transformados

Verificamos la calidad de los datos después de las transformaciones y validamos que cumplan con nuestros requisitos.

In [ ]:
# Validación completa de los datos transformados
print("✅ VALIDACIÓN DE DATOS TRANSFORMADOS")
print("=" * 60)

# 1. Resumen general de la transformación
print("1️⃣ RESUMEN DE LA TRANSFORMACIÓN:")
print("-" * 40)
final_count = df_clean.count()
final_columns = len(df_clean.columns)

print(f"   • Filas finales: {final_count:,}")
print(f"   • Columnas finales: {final_columns}")
print(f"   • Particiones: {df_clean.rdd.getNumPartitions()}")

# 2. Validación de valores nulos en datos limpios
print(f"\n2️⃣ VALIDACIÓN DE NULOS POST-TRANSFORMACIÓN:")
print("-" * 40)
for column in df_clean.columns:
    null_count = df_clean.filter(col(column).isNull()).count()
    null_percentage = (null_count / final_count) * 100
    status = "✅" if null_percentage == 0 else "⚠️" if null_percentage < 5 else "❌"
    print(f"   {status} {column}: {null_count:,} nulos ({null_percentage:.2f}%)")

# 3. Validación de rangos y tipos de datos
print(f"\n3️⃣ VALIDACIÓN DE RANGOS Y TIPOS:")
print("-" * 40)

# Validar que Label esté en el rango correcto
if 'Label' in df_clean.columns:
    label_values = df_clean.select('Label').distinct().collect()
    label_list = [row['Label'] for row in label_values]
    
    if all(label in [0, 1] for label in label_list if label is not None):
        print(f"   ✅ Label: valores válidos {label_list}")
    else:
        print(f"   ⚠️  Label: valores inesperados {label_list}")

# Validar longitudes de texto
text_length_cols = [col_name for col_name in df_clean.columns if col_name.endswith('_length')]
for length_col in text_length_cols:
    min_len = df_clean.agg(min(length_col)).collect()[0][0]
    max_len = df_clean.agg(max(length_col)).collect()[0][0]
    avg_len = df_clean.agg(avg(length_col)).collect()[0][0]
    
    if min_len >= 0 and max_len > 0:
        print(f"   ✅ {length_col}: rango válido [min:{min_len}, max:{max_len}, avg:{avg_len:.1f}]")
    else:
        print(f"   ⚠️  {length_col}: rango inválido [min:{min_len}, max:{max_len}]")

# 4. Validación de consistencia
print(f"\n4️⃣ VALIDACIÓN DE CONSISTENCIA:")
print("-" * 40)

# Verificar que no hay IDs duplicados
if 'row_id' in df_clean.columns:
    unique_ids = df_clean.select('row_id').distinct().count()
    total_rows = df_clean.count()
    
    if unique_ids == total_rows:
        print(f"   ✅ row_id: todos los IDs son únicos ({unique_ids:,})")
    else:
        print(f"   ❌ row_id: IDs duplicados detectados ({unique_ids:,} únicos de {total_rows:,} total)")

print(f"\n✅ Validación completada!")

# Mostrar muestra de datos transformados
print(f"\n📋 MUESTRA DE DATOS TRANSFORMADOS:")
print("-" * 40)
df_clean.select(*df_clean.columns[:8]).show(3, truncate=True)

## 📊 7. Visualizaciones y Análisis Gráfico

Creamos visualizaciones comprehensivas usando matplotlib, seaborn y plotly para analizar los datos transformados.

In [ ]:
# Convertir datos de Spark a Pandas para visualizaciones
print("📊 PREPARANDO DATOS PARA VISUALIZACIONES")
print("=" * 60)

print("🔄 Convirtiendo Spark DataFrame a Pandas...")

# Tomar una muestra para visualizaciones (si el dataset es muy grande)
sample_size = min(50000, df_clean.count())  # Máximo 50k filas para visualizaciones
sampling_fraction = sample_size / df_clean.count()

if sampling_fraction < 1.0:
    print(f"   📊 Tomando muestra de {sample_size:,} filas ({sampling_fraction:.2%} del dataset)")
    df_vis = df_clean.sample(False, sampling_fraction, seed=42)
else:
    print(f"   📊 Usando dataset completo ({sample_size:,} filas)")
    df_vis = df_clean

# Convertir a Pandas
df_pandas = df_vis.toPandas()

print(f"✅ Datos preparados para visualización:")
print(f"   • Filas: {len(df_pandas):,}")
print(f"   • Columnas: {len(df_pandas.columns)}")
print(f"   • Memoria estimada: {df_pandas.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Configuración de visualizaciones
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print(f"\n🎨 Configuración de matplotlib aplicada")

In [ ]:
# 1. Distribución de sentimientos
print("1️⃣ GRÁFICA DE DISTRIBUCIÓN DE SENTIMIENTOS")
print("-" * 50)

if 'Label' in df_pandas.columns or 'Label_normalized' in df_pandas.columns:
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Gráfica de barras
    if 'Label_normalized' in df_pandas.columns:
        sentiment_counts = df_pandas['Label_normalized'].value_counts()
        label_column = 'Label_normalized'
    else:
        sentiment_counts = df_pandas['Label'].value_counts()
        label_column = 'Label'
    
    # Subplot 1: Gráfica de barras
    bars = axes[0].bar(sentiment_counts.index, sentiment_counts.values, 
                       color=['#ff6b6b', '#4ecdc4'], alpha=0.8)
    axes[0].set_title('Distribución de Sentimientos', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Sentimiento')
    axes[0].set_ylabel('Número de Registros')
    
    # Añadir valores en las barras
    for bar in bars:
        height = bar.get_height()
        axes[0].annotate(f'{int(height):,}',
                        xy=(bar.get_x() + bar.get_width() / 2, height),
                        xytext=(0, 3),  
                        textcoords="offset points",
                        ha='center', va='bottom', fontweight='bold')
    
    # Subplot 2: Gráfica de pastel
    colors = ['#ff6b6b', '#4ecdc4']
    wedges, texts, autotexts = axes[1].pie(sentiment_counts.values, 
                                          labels=sentiment_counts.index,
                                          autopct='%1.1f%%', 
                                          colors=colors,
                                          explode=(0.05, 0.05))
    axes[1].set_title('Proporción de Sentimientos', fontsize=14, fontweight='bold')
    
    # Mejorar el texto del pie
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
    
    plt.tight_layout()
    plt.show()
    
    # Mostrar estadísticas
    total = sentiment_counts.sum()
    print(f"\n📊 Estadísticas de Sentimientos:")
    for sentiment, count in sentiment_counts.items():
        percentage = (count / total) * 100
        print(f"   • {sentiment}: {count:,} registros ({percentage:.1f}%)")
        
else:
    print("⚠️  Columna de sentimientos no encontrada")

In [ ]:
# 2. Análisis temporal (si hay columnas de fecha)
print("\n2️⃣ ANÁLISIS TEMPORAL")
print("-" * 50)

date_columns = [col for col in df_pandas.columns if 'date' in col.lower() and not col.endswith('_year') 
                and not col.endswith('_month') and not col.endswith('_day')]

if date_columns:
    date_col = date_columns[0]  # Usar la primera columna de fecha encontrada
    
    # Verificar si tenemos columnas derivadas
    year_col = f"{date_col}_year"
    month_col = f"{date_col}_month"
    weekday_col = f"{date_col}_weekday"
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Gráfica 1: Distribución por año (si existe)
    if year_col in df_pandas.columns:
        year_counts = df_pandas[year_col].value_counts().sort_index()
        axes[0, 0].plot(year_counts.index, year_counts.values, marker='o', linewidth=2, markersize=6)
        axes[0, 0].set_title('Distribución de Registros por Año', fontweight='bold')
        axes[0, 0].set_xlabel('Año')
        axes[0, 0].set_ylabel('Número de Registros')
        axes[0, 0].grid(True, alpha=0.3)
    
    # Gráfica 2: Distribución por mes
    if month_col in df_pandas.columns:
        month_counts = df_pandas[month_col].value_counts().sort_index()
        month_names = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun',
                      'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
        axes[0, 1].bar(range(1, len(month_counts)+1), month_counts.values, color='skyblue', alpha=0.8)
        axes[0, 1].set_title('Distribución de Registros por Mes', fontweight='bold')
        axes[0, 1].set_xlabel('Mes')
        axes[0, 1].set_ylabel('Número de Registros')
        if len(month_counts) <= 12:
            axes[0, 1].set_xticks(range(1, len(month_counts)+1))
            axes[0, 1].set_xticklabels([month_names[i-1] for i in month_counts.index], rotation=45)
    
    # Gráfica 3: Sentimiento por día de la semana
    if weekday_col in df_pandas.columns and ('Label' in df_pandas.columns or 'Label_normalized' in df_pandas.columns):
        sentiment_col = 'Label_normalized' if 'Label_normalized' in df_pandas.columns else 'Label'
        
        weekday_sentiment = pd.crosstab(df_pandas[weekday_col], df_pandas[sentiment_col])
        weekday_sentiment.plot(kind='bar', ax=axes[1, 0], color=['#ff6b6b', '#4ecdc4'])
        axes[1, 0].set_title('Sentimientos por Día de la Semana', fontweight='bold')
        axes[1, 0].set_xlabel('Día de la Semana')
        axes[1, 0].set_ylabel('Número de Registros')
        axes[1, 0].tick_params(axis='x', rotation=45)
        axes[1, 0].legend(title='Sentimiento')
    
    # Gráfica 4: Heatmap de sentimiento por mes y año
    if year_col in df_pandas.columns and month_col in df_pandas.columns and ('Label' in df_pandas.columns or 'Label_normalized' in df_pandas.columns):
        sentiment_col = 'Label_normalized' if 'Label_normalized' in df_pandas.columns else 'Label'
        
        # Crear tabla pivote
        pivot_data = df_pandas.groupby([year_col, month_col]).size().unstack(fill_value=0)
        
        if len(pivot_data) > 0:
            sns.heatmap(pivot_data.T, annot=True, fmt='d', cmap='YlOrRd', ax=axes[1, 1])
            axes[1, 1].set_title('Heatmap: Registros por Mes y Año', fontweight='bold')
            axes[1, 1].set_xlabel('Año')
            axes[1, 1].set_ylabel('Mes')
    
    plt.tight_layout()
    plt.show()
    
    print(f"✅ Análisis temporal completado usando columna: {date_col}")
    
else:
    print("⚠️  No se encontraron columnas de fecha para análisis temporal")

In [ ]:
# 3. Análisis de longitud de texto
print("\n3️⃣ ANÁLISIS DE LONGITUD DE TEXTO")
print("-" * 50)

text_length_cols = [col for col in df_pandas.columns if col.endswith('_length')]
word_count_cols = [col for col in df_pandas.columns if col.endswith('_word_count')]

if text_length_cols or word_count_cols:
    
    # Determinar número de subplots necesarios
    n_plots = len(text_length_cols) + len(word_count_cols)
    n_cols = min(2, n_plots)
    n_rows = (n_plots + 1) // 2
    
    if n_plots > 0:
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
        if n_plots == 1:
            axes = [axes]
        elif n_rows == 1:
            axes = axes if isinstance(axes, np.ndarray) else [axes]
        else:
            axes = axes.flatten()
        
        plot_idx = 0
        
        # Gráficas de longitud de caracteres
        for length_col in text_length_cols:
            if plot_idx < len(axes):
                # Histograma de longitudes
                axes[plot_idx].hist(df_pandas[length_col].dropna(), bins=30, 
                                   color='lightblue', alpha=0.7, edgecolor='black')
                axes[plot_idx].set_title(f'Distribución de {length_col.replace("_", " ").title()}', 
                                        fontweight='bold')
                axes[plot_idx].set_xlabel('Longitud (caracteres)')
                axes[plot_idx].set_ylabel('Frecuencia')
                axes[plot_idx].grid(True, alpha=0.3)
                
                # Añadir estadísticas
                mean_length = df_pandas[length_col].mean()
                median_length = df_pandas[length_col].median()
                axes[plot_idx].axvline(mean_length, color='red', linestyle='--', 
                                      label=f'Media: {mean_length:.1f}')
                axes[plot_idx].axvline(median_length, color='green', linestyle='--', 
                                      label=f'Mediana: {median_length:.1f}')
                axes[plot_idx].legend()
                
                plot_idx += 1
        
        # Gráficas de conteo de palabras
        for word_col in word_count_cols:
            if plot_idx < len(axes):
                # Histograma de conteo de palabras
                axes[plot_idx].hist(df_pandas[word_col].dropna(), bins=30, 
                                   color='lightcoral', alpha=0.7, edgecolor='black')
                axes[plot_idx].set_title(f'Distribución de {word_col.replace("_", " ").title()}', 
                                        fontweight='bold')
                axes[plot_idx].set_xlabel('Número de Palabras')
                axes[plot_idx].set_ylabel('Frecuencia')
                axes[plot_idx].grid(True, alpha=0.3)
                
                # Añadir estadísticas
                mean_words = df_pandas[word_col].mean()
                median_words = df_pandas[word_col].median()
                axes[plot_idx].axvline(mean_words, color='red', linestyle='--', 
                                      label=f'Media: {mean_words:.1f}')
                axes[plot_idx].axvline(median_words, color='green', linestyle='--', 
                                      label=f'Mediana: {median_words:.1f}')
                axes[plot_idx].legend()
                
                plot_idx += 1
        
        # Ocultar axes no utilizados
        for i in range(plot_idx, len(axes)):
            axes[i].set_visible(False)
        
        plt.tight_layout()
        plt.show()
        
        # Mostrar estadísticas detalladas
        print(f"\n📊 Estadísticas de Texto:")
        for col in text_length_cols + word_count_cols:
            if col in df_pandas.columns:
                stats = df_pandas[col].describe()
                print(f"\n   📝 {col.replace('_', ' ').title()}:")
                print(f"      • Promedio: {stats['mean']:.2f}")
                print(f"      • Mediana: {stats['50%']:.2f}")
                print(f"      • Mínimo: {stats['min']:.0f}")
                print(f"      • Máximo: {stats['max']:.0f}")
                print(f"      • Desviación estándar: {stats['std']:.2f}")
else:
    print("⚠️  No se encontraron columnas de análisis de texto")

In [ ]:
# 4. Gráfica interactiva con Plotly
print("\n4️⃣ VISUALIZACIÓN INTERACTIVA CON PLOTLY")
print("-" * 50)

# Crear visualización interactiva si tenemos datos de sentimiento y texto
text_length_col = next((col for col in df_pandas.columns if col.endswith('_length')), None)
sentiment_col = 'Label_normalized' if 'Label_normalized' in df_pandas.columns else 'Label' if 'Label' in df_pandas.columns else None

if text_length_col and sentiment_col:
    
    # Crear scatter plot interactivo
    fig = px.scatter(df_pandas.sample(n=min(1000, len(df_pandas))), 
                     x=text_length_col, 
                     y='row_id' if 'row_id' in df_pandas.columns else range(len(df_pandas)),
                     color=sentiment_col,
                     title='Análisis Interactivo: Longitud de Texto vs Sentimiento',
                     labels={
                         text_length_col: 'Longitud del Texto (caracteres)',
                         'row_id': 'ID del Registro',
                         sentiment_col: 'Sentimiento'
                     },
                     hover_data=[sentiment_col, text_length_col])
    
    fig.update_layout(
        width=900,
        height=600,
        title_font_size=16,
        showlegend=True
    )
    
    fig.show()
    
    # Crear box plot interactivo
    fig2 = px.box(df_pandas, 
                  x=sentiment_col, 
                  y=text_length_col,
                  title='Distribución de Longitud de Texto por Sentimiento',
                  labels={
                      text_length_col: 'Longitud del Texto (caracteres)',
                      sentiment_col: 'Sentimiento'
                  })
    
    fig2.update_layout(
        width=800,
        height=500,
        title_font_size=16
    )
    
    fig2.show()
    
    print(f"✅ Visualizaciones interactivas creadas")
    print(f"   📊 Variables utilizadas: {sentiment_col} y {text_length_col}")
    
else:
    print("⚠️  No se pueden crear gráficas interactivas (faltan columnas requeridas)")
    print(f"   Columnas disponibles: {list(df_pandas.columns)[:10]}...")  # Mostrar las primeras 10

## 💾 8. Guardado de Datos Procesados (Load)

Guardamos los datos transformados en múltiples formatos para diferentes casos de uso, aprovechando las capacidades de escritura de PySpark.

In [ ]:
# Configuración de rutas de salida
print("💾 GUARDANDO DATOS PROCESADOS")
print("=" * 60)

output_base_path = "Files/"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Crear directorio de salida si no existe
os.makedirs(output_base_path, exist_ok=True)
os.makedirs(os.path.join(output_base_path, "processed"), exist_ok=True)

print("1️⃣ GUARDANDO EN FORMATO PARQUET (RECOMENDADO PARA BIG DATA)...")
print("-" * 50)

try:
    parquet_path = f"{output_base_path}processed/stock_sentiment_processed.parquet"
    
    # Guardar en formato Parquet (optimizado para analytics)
    df_clean.coalesce(1).write \
        .mode("overwrite") \
        .option("compression", "snappy") \
        .parquet(parquet_path)
    
    print(f"   ✅ Datos guardados en Parquet: {parquet_path}")
    print(f"   📊 Compresión: Snappy (balance entre velocidad y tamaño)")
    
except Exception as e:
    print(f"   ❌ Error guardando Parquet: {str(e)}")

print(f"\n2️⃣ GUARDANDO EN FORMATO CSV...")
print("-" * 50)

try:
    csv_path = f"{output_base_path}stock_sentiment_processed_{timestamp}.csv"
    
    # Guardar en CSV (para compatibilidad)
    df_clean.coalesce(1).write \
        .mode("overwrite") \
        .option("header", "true") \
        .csv(f"{output_base_path}processed/csv_output")
    
    # Renombrar el archivo CSV generado por Spark
    import glob
    csv_files = glob.glob(f"{output_base_path}processed/csv_output/part-*.csv")
    if csv_files:
        os.rename(csv_files[0], csv_path)
        # Limpiar directorio temporal
        import shutil
        shutil.rmtree(f"{output_base_path}processed/csv_output")
        
        print(f"   ✅ Datos guardados en CSV: {csv_path}")
    
except Exception as e:
    print(f"   ❌ Error guardando CSV: {str(e)}")

print(f"\n3️⃣ GUARDANDO METADATOS Y ESTADÍSTICAS...")
print("-" * 50)

try:
    # Crear reporte de metadatos
    metadata = {
        "timestamp": datetime.now().isoformat(),
        "total_records": df_clean.count(),
        "total_columns": len(df_clean.columns),
        "columns": df_clean.columns,
        "schema": str(df_clean.schema),
        "partitions": df_clean.rdd.getNumPartitions()
    }
    
    # Estadísticas adicionales
    if 'Label' in df_clean.columns:
        label_stats = df_clean.groupBy('Label').count().collect()
        metadata["label_distribution"] = {str(row['Label']): row['count'] for row in label_stats}
    
    # Guardar metadatos como JSON
    import json
    metadata_path = f"{output_base_path}processed/metadata_{timestamp}.json"
    
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2, default=str)
    
    print(f"   ✅ Metadatos guardados: {metadata_path}")
    
except Exception as e:
    print(f"   ❌ Error guardando metadatos: {str(e)}")

print(f"\n✅ PROCESO DE GUARDADO COMPLETADO!")
print(f"📁 Archivos generados en: {output_base_path}")
print(f"🕐 Timestamp: {timestamp}")